# Evaluating cellranger results

In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import os
import glob

In [25]:

# Get all CSV files
file_list = glob.glob("/home/raquelcr/cellranger_output/out_HetGla_female_1/NMR*/outs/metrics_summary.csv")

# Read and merge all files at once (vertically)
df = pd.concat([pd.read_csv(f) for f in file_list], ignore_index=True)
for col in df.columns:
    # Only process string/object columns
    if df[col].dtype == "object":
        
        # Remove surrounding spaces
        s = df[col].astype(str).str.strip()

        # Detect percentage columns
        if s.str.contains(r"%").any():
            df[col] = (
                s.str.replace("%", "", regex=False)
                 .str.replace(",", "", regex=False)
                 .astype(float) / 100
            )

        # Detect numeric columns with commas
        elif s.str.match(r"^-?[\d,]+(\.\d+)?$").all():
            df[col] = (
                s.str.replace(",", "", regex=False)
                 .astype(int)
            )

In [26]:

df['region'] = ['cerebral_cortex', 'cerebral_cortex', 'hippocampus','hippocampus','midbrain','midbrain']
df['sample']= [f'NMR{n}' for n in range(1,7)]


In [27]:
df.columns

Index(['Estimated Number of Cells', 'Mean Reads per Cell',
       'Median Genes per Cell', 'Number of Reads', 'Valid Barcodes',
       'Sequencing Saturation', 'Q30 Bases in Barcode',
       'Q30 Bases in RNA Read', 'Q30 Bases in UMI', 'Reads Mapped to Genome',
       'Reads Mapped Confidently to Genome',
       'Reads Mapped Confidently to Intergenic Regions',
       'Reads Mapped Confidently to Intronic Regions',
       'Reads Mapped Confidently to Exonic Regions',
       'Reads Mapped Confidently to Transcriptome',
       'Reads Mapped Antisense to Gene', 'Fraction Reads in Cells',
       'Total Genes Detected', 'Median UMI Counts per Cell', 'region',
       'sample'],
      dtype='object')

In [28]:
df.to_csv('cellranger_metrics.csv')
df

,Estimated Number of Cells,Mean Reads per Cell,Median Genes per Cell,Number of Reads,Valid Barcodes,Sequencing Saturation,Q30 Bases in Barcode,Q30 Bases in RNA Read,Q30 Bases in UMI,Reads Mapped to Genome,...,Reads Mapped Confidently to Intergenic Regions,Reads Mapped Confidently to Intronic Regions,Reads Mapped Confidently to Exonic Regions,Reads Mapped Confidently to Transcriptome,Reads Mapped Antisense to Gene,Fraction Reads in Cells,Total Genes Detected,Median UMI Counts per Cell,region,sample
0,14598,30105,1114,439471267,0.908,0.375,0.969,0.947,0.963,0.871,...,0.231,0.528,0.073,0.186,0.414,0.559,17971,1545,cerebral_cortex,NMR1
1,23933,38528,1054,922085392,0.900,0.518,0.970,0.944,0.964,0.917,...,0.241,0.560,0.077,0.199,0.438,0.559,18479,1420,cerebral_cortex,NMR2
2,7626,281612,1581,2147575086,0.882,0.845,0.969,0.933,0.964,0.899,...,0.234,0.547,0.083,0.236,0.394,0.439,18345,2354,hippocampus,NMR3
3,8660,107203,1224,928374135,0.891,0.759,0.969,0.942,0.964,0.878,...,0.232,0.527,0.081,0.247,0.361,0.437,18072,1724,hippocampus,NMR4
4,10000,201188,106,2011875667,0.936,0.993,0.968,0.951,0.961,0.944,...,0.252,0.481,0.179,0.568,0.091,0.454,15284,115,midbrain,NMR5
5,10000,197071,86,1970706119,0.927,0.995,0.968,0.943,0.961,0.942,...,0.249,0.482,0.176,0.572,0.085,0.167,14721,92,midbrain,NMR6


In [36]:
df_by_region = df.groupby('region')[['Estimated Number of Cells', 'Number of Reads']].sum()
df_by_region.T

region,cerebral_cortex,hippocampus,midbrain
Estimated Number of Cells,38531,16286,20000
Number of Reads,1361556659,3075949221,3982581786


In [ ]:
df.groupby('region')[['Mean Reads per Cell',
       'Median Genes per Cell', 'Valid Barcodes',
       'Sequencing Saturation', 'Q30 Bases in Barcode',
       'Q30 Bases in RNA Read', 'Q30 Bases in UMI', 'Reads Mapped to Genome',
       'Reads Mapped Confidently to Genome',
       'Reads Mapped Confidently to Intergenic Regions',
       'Reads Mapped Confidently to Intronic Regions',
       'Reads Mapped Confidently to Exonic Regions',
       'Reads Mapped Confidently to Transcriptome',
       'Reads Mapped Antisense to Gene', 'Fraction Reads in Cells',
       'Total Genes Detected',
       'Median UMI Counts per Cell']].mean()

In [35]:
formatted_df = df_by_region.copy()

for col in formatted_df.columns:

    if pd.api.types.is_float_dtype(formatted_df[col]):

        if formatted_df[col].dropna().between(0, 1).all():
            # Percent formatting
            formatted_df[col] = formatted_df[col].map(
                lambda x: f"{x:.1%}" if pd.notna(x) else ""
            )

        else:
            # Large number formatting
            formatted_df[col] = formatted_df[col].map(
                lambda x: f"{x:,.0f}" if pd.notna(x) else ""
            )

formatted_df.T

region,cerebral_cortex,hippocampus,midbrain
Mean Reads per Cell,"34,316","194,408","199,130"
Median Genes per Cell,"1,084","1,402",96
Valid Barcodes,90.4%,88.6%,93.2%
Sequencing Saturation,44.6%,80.2%,99.4%
Q30 Bases in Barcode,97.0%,96.9%,96.8%
Q30 Bases in RNA Read,94.5%,93.8%,94.7%
Q30 Bases in UMI,96.4%,96.4%,96.1%
Reads Mapped to Genome,89.4%,88.9%,94.3%
Reads Mapped Confidently to Genome,85.5%,85.2%,91.0%
Reads Mapped Confidently to Intergenic Regions,23.6%,23.3%,25.1%


## Cell Bender Results

In [22]:

# Get all CSV files
file_list = glob.glob("/home/raquelcr/NMR-snRNA-seq/cellbender/denoised/NMR*_metrics.csv")

# Read and merge all files at once (vertically)

df = pd.concat([pd.read_csv(f,header=None,names=["metric", ""]).set_index('metric').T for f in file_list], ignore_index=True)

df['region'] = ['cerebral_cortex', 'cerebral_cortex', 'hippocampus','hippocampus','midbrain','midbrain']
df['sample']= [f'NMR{n}' for n in range(1,7)]
for col in df.columns:
    # Only process string/object columns
    if df[col].dtype == "object":
        
        # Remove surrounding spaces
        s = df[col].astype(str).str.strip()

        # Detect percentage columns
        if s.str.contains(r"%").any():
            df[col] = (
                s.str.replace("%", "", regex=False)
                 .str.replace(",", "", regex=False)
                 .astype(float) / 100
            )

        # Detect numeric columns with commas
        elif s.str.match(r"^-?[\d,]+(\.\d+)?$").all():
            df[col] = (
                s.str.replace(",", "", regex=False)
                 .astype(int)
            )



In [23]:
df.columns

Index(['total_raw_counts', 'total_output_counts', 'total_counts_removed',
       'fraction_counts_removed', 'total_raw_counts_in_cells',
       'total_counts_removed_from_cells', 'fraction_counts_removed_from_cells',
       'average_counts_removed_per_cell', 'target_fpr', 'expected_cells',
       'found_cells', 'output_average_counts_per_cell',
       'ratio_of_found_cells_to_expected_cells', 'found_empties',
       'fraction_of_analyzed_droplets_that_are_nonempty',
       'convergence_indicator', 'overall_change_in_train_elbo', 'region',
       'sample'],
      dtype='object', name='metric')

In [28]:
df1 = df.groupby('region').sum()[['total_raw_counts', 'total_output_counts', 'total_counts_removed', 'total_counts_removed_from_cells', 'total_raw_counts_in_cells',
       'expected_cells', 'found_cells', 'found_empties']].T

In [48]:
df2 = df.groupby('region')[['fraction_counts_removed',
       'fraction_counts_removed_from_cells',
       'average_counts_removed_per_cell', 'target_fpr', 'output_average_counts_per_cell',
       'ratio_of_found_cells_to_expected_cells',
       'fraction_of_analyzed_droplets_that_are_nonempty',
       'convergence_indicator', 'overall_change_in_train_elbo']].mean().T
df2

region,cerebral_cortex,hippocampus,midbrain
metric,,,
fraction_counts_removed,0.3295,0.1210,0.2135
fraction_counts_removed_from_cells,0.3295,0.1210,0.2135
average_counts_removed_per_cell,656.9655,385.3400,43.6525
target_fpr,0.0100,0.0100,0.0100
output_average_counts_per_cell,1290.2785,2846.4280,188.7555
ratio_of_found_cells_to_expected_cells,2.0395,2.4265,2.3805
fraction_of_analyzed_droplets_that_are_nonempty,0.8435,0.3730,0.4360
convergence_indicator,0.4670,0.9480,1.2495
overall_change_in_train_elbo,705.6535,637.7315,42.9360


In [49]:
df2 = pd.concat([df1,df2])

In [50]:
df2

region,cerebral_cortex,hippocampus,midbrain
metric,,,
total_raw_counts,6.642773e+07,4.733452e+07,3.799279e+06
total_output_counts,4.285813e+07,4.165016e+07,3.052350e+06
total_counts_removed,2.356961e+07,5.684354e+06,7.469290e+05
total_counts_removed_from_cells,2.356960e+07,5.684348e+06,7.469290e+05
total_raw_counts_in_cells,6.642772e+07,4.733451e+07,3.799279e+06
expected_cells,1.676000e+04,6.186000e+03,7.768000e+03
found_cells,3.375500e+04,1.492200e+04,1.744900e+04
found_empties,6.245000e+03,2.507800e+04,2.255100e+04
fraction_counts_removed,3.295000e-01,1.210000e-01,2.135000e-01


In [53]:
formatted_df = df2.T.copy()

for col in formatted_df.columns:

    if pd.api.types.is_float_dtype(formatted_df[col]):

        if not formatted_df[col].dropna().between(0, 10).all():
            # Large number formatting
            formatted_df[col] = formatted_df[col].map(
                lambda x: f"{x:,.0f}" if pd.notna(x) else ""
            )

formatted_df.T

region,cerebral_cortex,hippocampus,midbrain
metric,,,
total_raw_counts,"66,427,732","47,334,516","3,799,279"
total_output_counts,"42,858,126","41,650,162","3,052,350"
total_counts_removed,"23,569,606","5,684,354","746,929"
total_counts_removed_from_cells,"23,569,598","5,684,348","746,929"
total_raw_counts_in_cells,"66,427,724","47,334,510","3,799,279"
expected_cells,"16,760","6,186","7,768"
found_cells,"33,755","14,922","17,449"
found_empties,"6,245","25,078","22,551"
fraction_counts_removed,0.3295,0.121,0.2135
